In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

In [2]:
DATA_DIR = os.path.dirname(os.path.abspath('datathon_v42.ipynb'))
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    # v24 BUG FIX: interval data is 2025, not 2024. year=2024 was wrong — caused all DOW labels
    # to be off by 1 day (e.g. April 1 2024=Monday but April 1 2025=Tuesday).
    df['Date'] = pd.to_datetime(dict(year=2025, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

# v42: impute 2024 staffing rows using 2025 DOW medians.
# staff only covers 2025 (365 days), so 2024 rows got NaN after left join in make_features().
# The ok=notna().all() filter in Cell 4 was silently dropping all 2024 training rows.
# Fix: fill 2024 NaNs with same-DOW median from 2025 data (staffing schedules are stable YoY).
staff['_dow'] = staff['Date'].dt.dayofweek
for _col in [f'Staff_{p}' for p in portfolios]:
    _med = staff[staff['Date'].dt.year >= 2025].groupby('_dow')[_col].median()
    _mask = staff[_col].isna()
    staff.loc[_mask, _col] = staff.loc[_mask, '_dow'].map(_med)
staff = staff.drop(columns='_dow')
print(f'staffing after imputation: {staff[[f"Staff_{p}" for p in portfolios]].isna().sum().sum()} NaNs remaining')

{'A': 731, 'B': 731, 'C': 731, 'D': 731}
{'A': 4076, 'B': 4285, 'C': 4359, 'D': 4358}
staffing: 365 days
staffing after imputation: 0 NaNs remaining


In [3]:
# features
holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])

def make_features(df, port):
    f = pd.DataFrame()
    f['Date'] = df['Date']
    f['dow'] = df['Date'].dt.dayofweek
    f['dom'] = df['Date'].dt.day
    f['month'] = df['Date'].dt.month
    f['woy'] = df['Date'].dt.isocalendar().week.astype(int)
    f['year'] = df['Date'].dt.year
    f['wknd'] = (f['dow'] >= 5).astype(int)
    f['mon'] = (f['dow'] == 0).astype(int)
    
    # tried one-hot encoding dow but this worked better
    f['dow_s'] = np.sin(2*np.pi*f['dow']/7)
    f['dow_c'] = np.cos(2*np.pi*f['dow']/7)
    f['month_s'] = np.sin(2*np.pi*f['month']/12)
    f['month_c'] = np.cos(2*np.pi*f['month']/12)
    
    f['holiday'] = df['Date'].isin(holidays).astype(int)
    f['month_start'] = (f['dom'] <= 5).astype(int)
    f['month_end'] = (f['dom'] >= 26).astype(int)
    
    # lags
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m not in df.columns: continue
        f[f'{m}_l7'] = df[m].shift(7)
        f[f'{m}_l14'] = df[m].shift(14)
        f[f'{m}_l28'] = df[m].shift(28)
        f[f'{m}_l365'] = df[m].shift(365)
        f[f'{m}_r7'] = df[m].rolling(7).mean()
        f[f'{m}_r30'] = df[m].rolling(30).mean()
        f[f'{m}_ew'] = df[m].ewm(span=7).mean()
    
    # staffing
    sc = f'Staff_{port}'
    f = f.merge(staff[['Date',sc]].rename(columns={sc:'agents'}), on='Date', how='left')
    
    for m in ['Call Volume','CCT','Abandon Rate']:
        if m in df.columns:
            f[f'tgt_{m}'] = df[m]
    return f

feats = {}
for p in portfolios:
    feats[p] = make_features(daily[p], p)
print({p: feats[p].shape for p in portfolios})


{'A': (731, 40), 'B': (731, 40), 'C': (731, 40), 'D': (731, 40)}


In [4]:
# v34: impute missing Call Volume slots with DOW+slot median instead of dropping.
# Fixes systematic bias: if specific slots (e.g. overnight) are always missing for certain DOWs,
# dropna never trains those patterns. Imputation also fixes daily_cv — incomplete days had
# artificially low daily_cv as a feature (sum of available slots only).
interval_models = {}
abd_prof = {}
cct_prof = {}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek

    # Build DOW+slot median lookup from observed (non-null) values
    imp = df.groupby(['dow','slot'])['Call Volume'].median()

    # Impute missing slots with their DOW+slot median
    missing = df['Call Volume'].isna()
    if missing.any():
        def fill(row):
            return imp.get((row['dow'], row['slot']), 0.0)
        df.loc[missing, 'Call Volume'] = df[missing].apply(fill, axis=1)
        print(f'{p}: imputed {missing.sum()} missing slots')
    else:
        print(f'{p}: no missing slots')

    # Recompute daily_cv from full 48-slot sum (now accurate for all days)
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','daily_cv']
    df = df.merge(dtot, on='Date')

    df['is_peak'] = ((df['slot']//2>=9)&(df['slot']//2<=17)).astype(int)
    df['slot_sin'] = np.sin(2*np.pi*df['slot']/48)
    df['slot_cos'] = np.cos(2*np.pi*df['slot']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
    df['month'] = df['Date'].dt.month
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)
    df['dom'] = df['Date'].dt.day
    df['dom_sin'] = np.sin(2*np.pi*df['dom']/31)
    df['dom_cos'] = np.cos(2*np.pi*df['dom']/31)

    icols = ['slot','dow','daily_cv','is_peak','slot_sin','slot_cos','dow_sin','dow_cos',
             'month_sin','month_cos','dom_sin','dom_cos']
    clean = df  # all rows now valid — no dropna needed

    hgb = HistGradientBoostingRegressor(max_iter=250, max_depth=4,
        learning_rate=0.05, min_samples_leaf=8, l2_regularization=1.0, random_state=42)
    hgb.fit(clean[icols].values, clean['Call Volume'].values)

    et = ExtraTreesRegressor(n_estimators=200, max_depth=8,
        min_samples_leaf=5, random_state=42, n_jobs=-1)
    et.fit(clean[icols].values, clean['Call Volume'].values)

    interval_models[p] = {'hgb': hgb, 'et': et, 'cols': icols}

    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    abd_prof[p], cct_prof[p] = {}, {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        pr = sub.groupby('slot')['abd_pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        if a.sum() > 0: a /= a.sum()
        abd_prof[p][dow] = a
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a, 0); a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a

prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2025-04-01') & (daily[p]['Date']<='2025-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()
print('done')


A: imputed 81 missing slots
B: imputed 100 missing slots
C: imputed 69 missing slots
D: imputed 73 missing slots
done


In [5]:
# HYBRID: actual daily CV + ML CCT (simpler, less overfit) + historical ABD

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    d = daily[p]
    aug_data = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2025)].sort_values('Date')
    preds[p] = {}
    preds[p]['Call Volume'] = aug_data['Call Volume'].values.copy()
    
    vals = preds[p]['Call Volume']
    if np.any(np.isnan(vals)):
        nan_idx = np.where(np.isnan(vals))[0]
        valid = aug_data[aug_data['Call Volume'].notna()]
        for idx in nan_idx:
            dow = aug[idx].dayofweek
            same_dow = valid[valid['Date'].dt.dayofweek==dow]['Call Volume']
            vals[idx] = same_dow.mean() if len(same_dow)>0 else valid['Call Volume'].mean()

def feat_cols(df):
    skip = ['Date','tgt_Call Volume','tgt_CCT','tgt_Abandon Rate']
    return [c for c in df.columns if c not in skip]

for p in portfolios:
    ft = feats[p]
    cols = feat_cols(ft)
    ok = ft[cols].notna().all(axis=1)
    cl = ft[ok].copy()
    trn = cl['Date'] < '2025-07-01'
    Xtr = cl.loc[trn, cols].values
    d = daily[p]
    a24 = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2024)]
    
    ytr = cl.loc[trn, 'tgt_CCT'].values
    # simpler CCT model â€” depth=3, more regularized, less overfit
    gb = HistGradientBoostingRegressor(loss='quantile', quantile=0.52,
        max_iter=200, max_depth=3, learning_rate=0.05,
        min_samples_leaf=15, l2_regularization=2.0, random_state=42)
    gb.fit(Xtr, ytr)
    # pure GB, no ridge blend (ridge hurts CCT)
    amsk = (ft['Date']>='2025-08-01') & (ft['Date']<='2025-08-31')
    Xa = ft.loc[amsk, cols].ffill().bfill().fillna(0)
    ml = gb.predict(Xa.values)
    
    g24 = d[(d['Date'].dt.year==2024)&(d['Date'].dt.month<=7)]['CCT'].mean()
    g25 = d[(d['Date'].dt.year==2025)&(d['Date'].dt.month<=7)]['CCT'].mean()
    gr = g25/g24 if g24>0 else 1.0
    bl = np.zeros(31)
    for i,dt in enumerate(aug):
        m = a24[a24['Date'].dt.dayofweek==dt.dayofweek]['CCT'].values
        bl[i] = m.mean()*gr if len(m)>0 else a24['CCT'].mean()*gr
    preds[p]['CCT'] = 0.7*ml[:31] + 0.3*bl
    
    recent = d[(d['Date']>='2025-06-01')&(d['Date']<'2025-08-01')]
    abd = np.zeros(31)
    for i,dt in enumerate(aug):
        dw = dt.dayofweek
        r = recent[recent['Date'].dt.dayofweek==dw]['Abandon Rate']
        a = a24[a24['Date'].dt.dayofweek==dw]['Abandon Rate']
        if len(r)>0 and len(a)>0: abd[i] = 0.6*r.mean() + 0.4*a.mean()
        elif len(r)>0: abd[i] = r.mean()
        else: abd[i] = d['Abandon Rate'].tail(60).mean()
    abd *= 1.1
    abd = np.clip(abd, 0.002, 0.25)
    preds[p]['Abandon Rate'] = abd

for p in portfolios:
    cv = preds[p]['Call Volume']
    print(f'{p}: actual CV={cv.sum():,.0f}, ML CCT={preds[p]["CCT"].mean():.1f}, hist ABD={preds[p]["Abandon Rate"].mean():.4f}')

A: actual CV=110,613, ML CCT=321.3, hist ABD=0.0120
B: actual CV=261,572, ML CCT=335.2, hist ABD=0.0176
C: actual CV=567,384, ML CCT=335.9, hist ABD=0.0108
D: actual CV=290,139, ML CCT=325.9, hist ABD=0.0144


In [6]:
# v23: identical to v21 except X_pred includes dom_sin/dom_cos for each August day
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

from scipy.ndimage import gaussian_filter1d as gf1d
# v35: build DOW proportion profiles for ALL portfolios (not just B)
all_profiles = {}
for q in portfolios:
    qdf = intervals[q].copy()
    qdf['dow'] = qdf['Date'].dt.dayofweek
    qtot = qdf.groupby('Date')['Call Volume'].sum()
    qdf = qdf.merge(qtot.rename('dtot').reset_index(), on='Date')
    qdf['pct'] = qdf['Call Volume'] / qdf['dtot'].replace(0, np.nan)
    all_profiles[q] = {}
    for dow in range(7):
        sub = qdf[qdf['dow']==dow]
        pr = sub.groupby('slot')['pct'].median()
        a = np.zeros(48); a[pr.index.astype(int)] = pr.values
        a = np.nan_to_num(a,0); a = gf1d(a, sigma=0.7)
        if a.sum()>0: a/=a.sum()
        all_profiles[q][dow] = a

# August month encoding (fixed for all prediction rows)
aug_month_sin = np.sin(2*np.pi*8/12)
aug_month_cos = np.cos(2*np.pi*8/12)

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    hgb = interval_models[p]['hgb']
    et = interval_models[p]['et']
    icols = interval_models[p]['cols']

    for i,dt in enumerate(aug_dates):
        dw = dt.dayofweek
        # v23: day-of-month encoding for this specific August day
        aug_dom = dt.day
        dom_sin = np.sin(2*np.pi*aug_dom/31)
        dom_cos = np.cos(2*np.pi*aug_dom/31)

        rows = []
        for slot in range(48):
            is_peak = 1 if 9 <= slot//2 <= 17 else 0
            rows.append([slot, dw, dcv[i], is_peak,
                        np.sin(2*np.pi*slot/48), np.cos(2*np.pi*slot/48),
                        np.sin(2*np.pi*dw/7), np.cos(2*np.pi*dw/7),
                        aug_month_sin, aug_month_cos,
                        dom_sin, dom_cos])
        X_pred = np.array(rows)

        # 50/50 ensemble (same as v18/v21)
        cv_model = 0.5*np.clip(hgb.predict(X_pred),0,None) + 0.5*np.clip(et.predict(X_pred),0,None)
        if cv_model.sum() > 0:
            cv_model = cv_model * (dcv[i] / cv_model.sum())

        # v35: apply 70/30 ML+profile blend to all portfolios (was B-only)
        cv_prof = dcv[i] * all_profiles[p][dw]
        cv = 0.4 * cv_model + 0.6 * cv_prof  # v38: 40/60 blend — binary search between 50/50 (best) and 30/70 (no improvement)

        ab = dabd[i] * abd_prof[p][dw]
        sc = dcct[i] / prof_cct_avg[p] if prof_cct_avg[p]>0 else 1.0
        cc = cct_prof[p][dw] * sc
        ar = np.where(cv>0, ab/cv, 0.0)
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

In [7]:
# v18: asymmetric bias â€” deliberately overpredict to exploit understaffing penalty
# Team012 insight: higher MAPE (16.58%) but much lower WMAE (35.07)
# Understaffing (underprediction) is penalized MORE, so bias upward

CV_BIAS = 1.05    # +5% upward bias on call volume
CCT_BIAS = 1.03   # +3% upward bias on CCT (handle time)

for p in portfolios:
    # Apply upward bias to CV
    res[p]['cv'] = res[p]['cv'] * CV_BIAS
    
    # Apply upward bias to CCT
    res[p]['cct'] = res[p]['cct'] * CCT_BIAS
    
    # Standard cleanup
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

print(f'Applied bias: CV={CV_BIAS}x, CCT={CCT_BIAS}x')
for p in portfolios:
    print(f'{p}: CV={res[p]["cv"].sum():,}, CCT_avg={res[p]["cct"].mean():.1f}')

Applied bias: CV=1.05x, CCT=1.03x
A: CV=116,119, CCT_avg=310.5
B: CV=274,653, CCT_avg=324.9
C: CV=595,763, CCT_avg=326.7
D: CV=304,652, CCT_avg=318.7


In [8]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(DATA_DIR, 'forecast_v42.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all assertions passed')
print(f'Saved: {out}')

all assertions passed
Saved: c:\Users\dokek\OneDrive\GitHub\datathon\forecast_v42.csv
